# 12 — mmBERT base

**The screen winner, and the one the tokenizer heuristic said should lose.** In the 800-row
fine-tuning screen (`07_encoder_bakeoff.ipynb`) mmBERT took Negative-F1 **0.3268** against
xlmr-base's 0.2343 — and was the only candidate whose point estimate fell outside the classical
comparator's CI [0.2106, 0.3125].

It won *despite* the worst fertility in the roster. That is the standing lesson from the bake-off
report: fertility screens for efficiency and for catastrophic failure (`[UNK]`), **not** for
downstream accuracy.

**Caveat carried forward:** the screen trained on 800 balanced rows and evaluated on a 95/5
distribution, so its precision (0.20) is not meaningful. This notebook trains on the real
distribution with a weighted loss instead.

**Tokenizer fertility** (tokens per word, measured on 300 dev tickets per language):

| english | sinhala | singlish | tamil | tamilish |
|---|---|---|---|---|
| 1.37 | 4.4 | 2.17 | 3.91 | 2.4 |

Shatters native Sinhala at 4.40 tokens/word — nearly 2.5x XLM-R. Watch the sinhala cell.

---

**Protocol.** Fit on `train` (8,500 ids), select the epoch on **dev** `negative_f1`. Test is not
opened here — only the winner of `30_encoder_leaderboard.ipynb` is refit on train+dev and scored
on test. Training code is `swiftbench.train_encoder`, shared with the other five notebooks so the
numbers land in one table.

In [ ]:
import sys, warnings, json
from pathlib import Path
warnings.filterwarnings("ignore")

REPO = Path.cwd()
while not (REPO / "ml" / "swiftbench").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "ml"))

import numpy as np, pandas as pd
import swiftbench as sb
from swiftbench import config, metrics, splits, train_encoder as te

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

AUTHOR = "sithija"
print("split sha:", splits.sha(), "| device:", te.device())

## Baseline to beat

The classical champion from `10_final_test_eval.ipynb`, on the same dev split. An encoder that
does not clear this is not worth its serving cost.

In [ ]:
CLASSICAL_DEV = 0.6144      # tfidf-svm / class_weight / multi, pooled dev negative_f1
CLASSICAL_TEST = 0.4572     # same model, pooled test

runs = sb.results.load_all("dev")
if not runs.empty and "family" in runs.columns:
    done = runs[(runs.task == "sentiment") & (runs.family == "encoder")]
    if not done.empty:
        display(done[["model", "arm", "eval_lang", "headline", "best_epoch", "train_seconds"]]
                .sort_values("headline", ascending=False))
print(f"classical dev  negative_f1 {CLASSICAL_DEV:.4f}")
print(f"classical test negative_f1 {CLASSICAL_TEST:.4f}")

## Fine-tune

`SMOKE = True` runs a 1,200-row sanity pass in about a minute. Set it to `False` for the real
run.

In [ ]:
SMOKE = True          # <- set False for the real run

EPOCHS = 3
BATCH_SIZE = 32
LR = 2e-5
ARM = "class_weight"  # weighted loss; `ros` and `none` are the other two arms

run = te.run(
    task="sentiment",
    model="mmbert",
    eval_lang="all",
    arm=ARM,
    portion="dev",
    fit_portion="train",
    epochs=1 if SMOKE else EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    subsample=1200 if SMOKE else None,
    author=AUTHOR,
    save=not SMOKE,
)
run.scores

In [ ]:
display(run.history)
print(f"selected epoch {run.scores['best_epoch']} of {run.scores['epochs']}  "
      f"({run.scores['train_seconds']/60:.1f} min on {run.scores['device']})")

## Where it fails

Selection metric alone hides the operating point. At 95%+ Neutral, a model can post a healthy
Negative-F1 while its precision makes escalation unusable — the bake-off's encoders all sat at
0.12-0.20 precision against 0.77-0.90 recall.

In [ ]:
ev = run.eval_frame.copy()
ev["pred"] = run.predictions
ev["p_negative"] = run.scores_positive

print("confusion (rows = truth)")
labels, cm = metrics.confusion(ev.sentiment, ev.pred, "sentiment")
display(pd.DataFrame(cm, index=labels, columns=labels))

print(f"\nprecision {run.scores['negative_precision']:.4f}   "
      f"recall {run.scores['negative_recall']:.4f}   "
      f"negative_f1 {run.scores['headline']:.4f}")

In [ ]:
# Per-language breakdown.
rows = []
for lang in config.LANGUAGES:
    m = (ev.language == lang)
    if not m.any():
        continue
    s = metrics.score(ev.sentiment[m], ev.pred[m], "sentiment")
    rows.append({"language": lang, **{k: v for k, v in s.items() if not isinstance(v, str)}})
per_lang = pd.DataFrame(rows)
display(per_lang[["language", "negative_f1", "negative_precision", "negative_recall",
                  "accuracy", "n_negative_true"]])

In [ ]:
# False negatives -- angry customers routed to an auto-reply. These are the expensive errors.
missed = ev[(ev.sentiment == "Negative") & (ev.pred != "Negative")]
print(f"{len(missed)} missed Negative rows of {(ev.sentiment == 'Negative').sum()}")
display(missed.sort_values("p_negative", ascending=False)[["language", "text", "p_negative"]].head(12))

## Verdict

Fill this in after the real run.

- Beats classical dev (0.6144) — yes / no, and by how much against the CI width of ~0.15.
- Where it wins and loses per language, especially the `ALL` cell.
- Whether precision is high enough for escalation, or whether it needs the lexicon layer from
  `20_technique_lexicon_correction.ipynb`.

Then record it in `30_encoder_leaderboard.ipynb`.